# Reverie: Predicting your star rating for movies it never trained on
### Temporal-holdout regression with richer features (Deep Learning, IE)

The earlier notebook asked a yes/no question (rate 3+?) from genres alone, and found genre is a weak signal. This one goes further, in the spirit of the bike-rental ANN (a **regression** task, loss = MSE, report RMSE):

- **Predict the actual star rating** (0.5 to 5) of each film.
- **Richer features** than genre: the 18 genres, TMDB's own rating / popularity / runtime / foreign-language flag, **director affinity** and **lead-actor affinity** (your average past rating for that person, learned only from pre-2020 films), and the top keyword tags.
- **Temporal holdout:** train on pre-2020 films, test on the hidden 2020+ window (never seen during training).
- **Baselines to beat:** predict the mean rating, and predict from TMDB's rating alone.

**Honest notes:** still one user; affinity features cold-start to the global mean for directors/actors you have never rated (common for brand-new films); and TMDB's own rating may carry most of the signal (which would mean "you rate what critics rate", a finding in itself). No leakage: all scaling, affinities and keyword vocab come from the training films only.

# STEP 0: Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from collections import Counter

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, classification_report

np.random.seed(42); tf.random.set_seed(42)

ML_GENRES = ["Action","Adventure","Animation","Children's","Comedy","Crime",
    "Documentary","Drama","Fantasy","Film-Noir","Horror","Musical","Mystery",
    "Romance","Sci-Fi","Thriller","War","Western"]
NUMERIC = ["vote_average","vote_count","popularity","runtime","is_foreign"]
TOP_K_KEYWORDS = 30

# STEP 1: Load the dataset (built by scripts/build_validation_dataset.py)

In [ ]:
df = pd.read_csv("../data/validation_dataset.csv")
df["year"] = pd.to_numeric(df["year"], errors="coerce")
df = df.dropna(subset=["year"]).reset_index(drop=True)
df["year"] = df["year"].astype(int)
for col in ["director","lead_actor","keywords"]:
    df[col] = df[col].fillna("")
print(df.shape)
df.head()

# STEP 2: Explore the target (the actual star rating)

In [ ]:
print("rating mean/std:", round(df.rating.mean(),2), "/", round(df.rating.std(),2))
fig, ax = plt.subplots(1,2,figsize=(11,3.5))
sns.histplot(df.rating, bins=10, ax=ax[0]); ax[0].set_title("Your star ratings")
sns.histplot(df.year, bins=30, ax=ax[1]); ax[1].axvline(2020,color="red",ls="--"); ax[1].set_title("Films by year (red = 2020 split)")
plt.tight_layout()

# STEP 3: Temporal split + leakage-free feature engineering
Pre-2020 trains the model; 2020+ is the locked test set. **Director/actor affinity** and the **keyword vocabulary** are computed from the pre-2020 training films only, so the 2020+ test set never leaks into the features.

In [ ]:
train_full = df[df.year < 2020].copy()
test = df[df.year >= 2020].copy()
print(f"train (pre-2020): {len(train_full)}   |   test (2020+): {len(test)}")

# --- affinities: average PAST rating per director / lead actor (training only) ---
global_mean = train_full.rating.mean()
dir_mean = train_full.groupby("director").rating.mean()
act_mean = train_full.groupby("lead_actor").rating.mean()

def affinity(table, key):
    return table.get(key, global_mean) if key else global_mean

for d in (train_full, test, df):
    d["dir_aff"] = d["director"].map(lambda k: affinity(dir_mean, k))
    d["act_aff"] = d["lead_actor"].map(lambda k: affinity(act_mean, k))

# --- keyword vocabulary: most common tags in the training films ---
counts = Counter()
for ks in train_full.keywords:
    counts.update(k for k in ks.split("|") if k)
top_kw = [k for k,_ in counts.most_common(TOP_K_KEYWORDS)]
kw_cols = [f"kw::{k}" for k in top_kw]
for d in (train_full, test):
    for k, col in zip(top_kw, kw_cols):
        d[col] = d.keywords.apply(lambda s: int(k in s.split("|")))

FEATURES = ML_GENRES + NUMERIC + ["dir_aff","act_aff"] + kw_cols
print(f"{len(FEATURES)} features: {len(ML_GENRES)} genres + {len(NUMERIC)} numeric + 2 affinity + {len(kw_cols)} keywords")

In [ ]:
tr, val = train_test_split(train_full, test_size=0.2, random_state=42)
sc = StandardScaler()
Xtr = sc.fit_transform(tr[FEATURES]);   ytr = tr.rating.values
Xval = sc.transform(val[FEATURES]);     yval = val.rating.values
Xte = sc.transform(test[FEATURES]);     yte = test.rating.values
print("train/val/test:", len(ytr), len(yval), len(yte))

# STEP 4: Baselines to beat
**(a) Mean rating** — predict the training mean for every film (the naive regression baseline).
**(b) TMDB rating only** — a linear regression on TMDB's own vote_average. If the neural net cannot beat this, the signal is just "you rate what the crowd rates".

In [ ]:
def rmse(y, p): return mean_squared_error(y, p) ** 0.5

# (a) mean baseline
pred_mean = np.full_like(yte, ytr.mean())
# (b) TMDB-rating-only linear baseline
lin = LinearRegression().fit(tr[["vote_average"]], ytr)
pred_tmdb = lin.predict(test[["vote_average"]])

print(f"Mean baseline      RMSE {rmse(yte, pred_mean):.3f}  MAE {mean_absolute_error(yte, pred_mean):.3f}")
print(f"TMDB-rating linear RMSE {rmse(yte, pred_tmdb):.3f}  MAE {mean_absolute_error(yte, pred_tmdb):.3f}")

# STEP 5: The neural network regressor (bike-rental pattern)
Stacked Dense layers with ReLU and dropout, a single **linear** output, `loss='mse'`, early stopping on validation loss.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(len(FEATURES),)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1),
])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.summary()

In [ ]:
early = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True)
hist = model.fit(Xtr, ytr, validation_data=(Xval, yval),
                 epochs=400, batch_size=16, callbacks=[early], verbose=0)
print("stopped after", len(hist.history["loss"]), "epochs")
plt.figure(figsize=(6,3.5))
plt.plot(hist.history["loss"], label="train")
plt.plot(hist.history["val_loss"], label="validation")
plt.xlabel("epoch"); plt.ylabel("MSE"); plt.legend(); plt.title("Training vs validation loss")

In [ ]:
pred_nn = model.predict(Xte, verbose=0).ravel()
print(f"Neural net         RMSE {rmse(yte, pred_nn):.3f}  MAE {mean_absolute_error(yte, pred_nn):.3f}")

# STEP 6: Compare on the held-out 2020+ films

In [ ]:
summary = pd.DataFrame([
    {"model":"mean baseline",      "RMSE":round(rmse(yte,pred_mean),3),  "MAE":round(mean_absolute_error(yte,pred_mean),3)},
    {"model":"TMDB rating (linear)","RMSE":round(rmse(yte,pred_tmdb),3), "MAE":round(mean_absolute_error(yte,pred_tmdb),3)},
    {"model":"neural net",         "RMSE":round(rmse(yte,pred_nn),3),    "MAE":round(mean_absolute_error(yte,pred_nn),3)},
]).set_index("model")
summary

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(11,4))
ax[0].scatter(yte, pred_nn, alpha=0.6)
ax[0].plot([0.5,5],[0.5,5],"r--"); ax[0].set_xlabel("actual rating"); ax[0].set_ylabel("predicted"); ax[0].set_title("Neural net: predicted vs actual (2020+)")
sns.histplot(pred_nn - yte, bins=20, ax=ax[1]); ax[1].set_title("Residuals (predicted - actual)")
plt.tight_layout()

**Bonus, the recommend decision.** Threshold the predicted rating at 3 and recover the yes/no "would I recommend it" question, to compare against the first notebook.

In [ ]:
yte_bin = (yte >= 3).astype(int)
pred_bin = (pred_nn >= 3).astype(int)
print(classification_report(yte_bin, pred_bin, zero_division=0))

# Takeaways
- A **regression** with richer features is the right shape for this: it predicts how much you will like a film, not just yes/no, and reports RMSE the way her bike-rental notebook does.
- The honest test is **beating the mean baseline and the TMDB-rating-only baseline**. If the net wins, the personal features (director/actor affinity, keywords) carry real signal beyond crowd taste. If it only ties the TMDB-rating baseline, the honest finding is that your ratings mostly track general acclaim plus a genre/keyword tilt.
- **Affinity cold-start:** brand-new films often have a director or lead you have never rated, so those features fall back to the global mean and contribute little, a real limitation worth stating.
- Everything stays in scope: regression ANN (bike-rental), MSE/RMSE, dropout + early stopping, train-only feature engineering (the "golden rule of scaling"), and beating a naive baseline.